In [2]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

import os
import getpass
3
if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass.getpass("Enter Groq API Key: ")
if not os.environ.get('GOOGLE_API_KEY'):
    os.environ['GOOGLE_API_KEY'] = getpass.getpass("Enter Google API Key: ")

Enter Groq API Key:  ········
Enter Google API Key:  ········


In [ ]:
%pip install -q pylate

In [3]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
dense_vectorstore = Chroma(
    embedding_function=embeddings,
    persist_directory=str(project_root / "data" / "chroma_naive_gemini"),
)

raw = dense_vectorstore.get(include=["documents"])
leaf_texts = raw["documents"]
print(f"{len(leaf_texts)} chunks pulled from the existing dense vectorstore")

1555 chunks pulled from the existing dense vectorstore


In [9]:
# Building the leaf.json
import json
raw = dense_vectorstore.get(include=["documents"])
with open(project_root / "data" / "leaf_texts.json", "w") as f:
    json.dump(raw["documents"], f)
print(f"Dumped {len(raw['documents'])} chunks")

Dumped 1555 chunks


In [ ]:
# Build the ColBERT index
from ragatouille import RAG

INDEX_NAME = "rag_lab_colbert"
colbert_index_dir = project_root / "data" / "colbert_index"

colbert = RAG.from_pretrained("colbert-ir/colbertv2.0")
index_path = colbert.index(
    collection=leaf_texts,
    index_name=INDEX_NAME,
    index_root=str(colbert_index_dir),
)
print("ColBERT index built at:", index_path)

In [ ]:
from rag_lab.retrieval.colbert_backend import ColBERTVectorStoreShim

colbert_vectorstore = ColBERTVectorStoreShim(index_path)

In [7]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen2.5:7b",
    base_url="http://<4050-laptop-ip>:11434",
    temperature=0,
)